<a href="https://colab.research.google.com/github/NikosMav/netflix-catalog-search/blob/main/netflix_dense_retrieval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Catalog retrieval walkthrough (sparse → dense → hybrid → rerank)

**Author:** Nikolaos (Nikos) Mavrapidis ([NikosMav](https://github.com/NikosMav))

Thin companion for the `retrieval` package. The CLI is the product surface; this notebook is not a second source of metrics.

**Source of truth for numbers:** [`results/eval_metrics.json`](results/eval_metrics.json) (regenerate with `python -m retrieval eval`). Full write-up: [`RETRIEVAL.md`](RETRIEVAL.md).

```bash
python -m retrieval query "war between vietnam and usa" --method bm25
python -m retrieval query "dark crime thriller set in Scandinavia" --method hybrid-rerank
python -m retrieval eval --failures
```

Methods available in the CLI (same package): Boolean, TF-IDF, **BM25**, dense MiniLM, hybrid RRF, **CPU ms-marco MiniLM rerank**. `hybrid-rerank` = CE over `hybrid(bm25+dense,meta)`.

This notebook is **not** a replacement for [`netflix_data_analysis.ipynb`](netflix_data_analysis.ipynb) (EDA + Boolean/TF-IDF case study).

In [ ]:
from pathlib import Path
import sys
import json
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "retrieval").exists() and (ROOT.parent / "retrieval").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from retrieval import (
    BM25Retriever,
    BooleanRetriever,
    CrossEncoderReranker,
    DenseRetriever,
    HybridRetriever,
    SparseTfidfRetriever,
    load_catalog,
)

catalog = load_catalog(ROOT / "data" / "netflix_titles.csv")
len(catalog), catalog.columns.tolist()[:8]

## Build retrievers

Boolean, TF-IDF, **BM25**, dense MiniLM, hybrid RRF (`bm25+dense` on `text_meta`), and optional CE rerank over that hybrid (same wiring as `python -m retrieval … --method hybrid-rerank`). First dense/CE call downloads models and may take a few minutes.

In [ ]:
boolean = BooleanRetriever(catalog)
tfidf = SparseTfidfRetriever(catalog, text_field="text")
bm25 = BM25Retriever(catalog, text_field="text")
bm25_meta = BM25Retriever(catalog, text_field="text_meta")
dense = DenseRetriever(catalog, text_field="text", show_progress=True)
dense_meta = DenseRetriever(catalog, text_field="text_meta", show_progress=True)
hybrid_bm25 = HybridRetriever(
    catalog, retrievers=[bm25_meta, dense_meta], name="hybrid(bm25+dense,meta)"
)
# Optional: same second stage as CLI hybrid-rerank (downloads ms-marco CE on first use)
# hybrid_rerank = CrossEncoderReranker(
#     catalog, base=hybrid_bm25, text_field="text_meta", candidate_k=50, name="hybrid+rerank"
# )
dense.embeddings.shape

## Same demo queries as the sparse notebook

In [ ]:
def side_by_side(query, top_k=8):
    frames = {
        "boolean": boolean.query(query, top_k)[["rank", "score", "title"]],
        "tf-idf": tfidf.query(query, top_k)[["rank", "score", "title"]],
        "bm25": bm25.query(query, top_k)[["rank", "score", "title"]],
        "dense": dense.query(query, top_k)[["rank", "score", "title"]],
        "hybrid(bm25+dense,meta)": hybrid_bm25.query(query, top_k)[["rank", "score", "title"]],
    }
    for name, frame in frames.items():
        print("=" * 20, name, "=" * 20)
        display(frame)

for q in ["war between vietnam and usa", "Mickey Mouse"]:
    print("#" * 72)
    print("QUERY:", q)
    side_by_side(q)

## Metrics (read committed JSON — do not treat this notebook as source of truth)

Load [`results/eval_metrics.json`](results/eval_metrics.json). To regenerate after code changes, run `python -m retrieval eval` (slow; downloads models). Do not invent or hand-edit numbers here.

In [ ]:
metrics_path = ROOT / "results" / "eval_metrics.json"
payload = json.loads(metrics_path.read_text(encoding="utf-8"))
print(f"n_queries={payload['n_queries']}  source={metrics_path}")
display(payload["metrics"])
# Optional full recompute (slow): from retrieval.evaluate import run_evaluation, results_to_markdown
# metrics = run_evaluation(show_progress=False); display(metrics)

## Limitations

Catalog search ≠ production recommender ≠ RAG-over-the-web. 28 author-labeled queries, demo models, no CIs. See [`RETRIEVAL.md`](RETRIEVAL.md).